In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('/projects/activities/kappsen-tmc/USERS/domans/differential-annotator-dev/image-differential-annotator/')

from annotator.annotation import runAnnotation
from annotator.combineCDF import getDiscreteCombinedCDFofAllFeatures as PCMA
from annotator.stqutils import loadAd, preparePatchesWSI, getPatchRepresentation, inferProb, showProb

from tqdm import tqdm
import numpy as np
import pandas as pd

# qs = np.linspace(0.05, 0.95, 10, endpoint=True)
qs = np.array([0.01, 0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.95, 0.99])

In [ ]:
# samples  = ['JAX_012_KD_C', 'JAX_013_KD_C', 'JAX_014_KD_C', 'JAX_015_KD_C']
# outsSTQpath = '/projects/activities/kappsen-tmc/USERS/domans/results-kidney-wedges/'

# samples  = ['MC_PLCP_0015', 'MC_PLBP_0015', 'MC_UC_0015', 'MC_PLACM_0015']
# outsSTQpath = '/projects/activities/kappsen-tmc/USERS/domans/results-placenta/'

# samples  = ['SA_001_L_C_a', 'SA_001_L_CM_w', 'SA_001_L_M_j', 'SA_001_L_P_mnop']
samples  = ['SA_001_L_C_a']
outsSTQpath = '/projects/activities/kappsen-tmc/USERS/domans/results-kidney-001/'

# samples = ['Image_0016903'] # JDC-WP-002-v
# outsSTQpath = '/projects/activities/kappsen-tmc/USERS/domans/results-STQ-post-xenium-32-final/'

In [ ]:
# If L is None, then the annotator will do lazy loading of patches with Zarr, else load entire images at requested level L
L = None
ts = 56
mpp = 0.25

# Load the STQ data for each sample
ads = {}
imgs = {}
for sample in tqdm(samples):
    ads[sample], imgs[sample] = loadAd(f'{outsSTQpath}{sample}/', fname='img.data.ctranspath-1.h5ad', L=L)

# Prepare the patches coordinates for each sample and concatenate them into a single DataFrame
patchCoordinates = pd.concat([preparePatchesWSI(ads[sample].obs, N=8, spacing=ts/mpp, sample_id=sample) for sample in tqdm(samples)], axis=0) # N=8

# Get the patch SAMPLER representations for each sample and combine them into a single DataFrame
patchesCDFs = pd.concat([getPatchRepresentation(ads[sample], patchCoordinates.xs(sample, level='sample', axis=0), qs, sample_id=sample) for sample in tqdm(samples)], axis=0)

In [ ]:
clf, plog, bp = {}, [], {}
runAnnotation(patchCoordinates, patchesCDFs, imgs, bp, clf, plog, ads=ads, qs=qs,
            figsize=(5,5), augFunc=PCMA, alpha=0.5, R=1, cmapColors=['lightcoral', 'blue'],
            seed=42, randomness=0.5, L=1 if L is None else L, sh=int(ts/2)/mpp)

In [ ]:
infSample  = samples[0]
x, y, p = inferProb(ads[infSample], clf['clf'], qs, tsize=ts/mpp, R=1)

In [ ]:
showProb(x, y, p, s=1, marker='s', figsize=(10, 5), colorbar=True, vmin=0.5, vmax=1, title=infSample, cmapColors=['lightcoral', 'blue'])